# Tarea 3: Análisis Cuantitativo de SBOMs y Vulnerabilidades

**Organización analizada**: [pallets](https://github.com/pallets)

3 repositorios de esta organización poseen commits de 1 mes atrás, lo que indica actividad reciente.

## Objetivo

Este notebook implementa el análisis cuantitativo de la Tarea 3, cubriendo las tres partes del enunciado:

- **Parte 1 — SBOMs**: Analiza los Software Bill of Materials generados por Syft para cada repositorio. Permite conocer cuántos componentes tiene cada proyecto, qué ecosistemas de paquetes utilizan (pip, npm, cargo, etc.) y cuáles son las dependencias más comunes entre repos.

- **Parte 2a — Vulnerabilidades en dependencias (SCA)**: Analiza los resultados de Grype, que escanea los SBOMs de la Parte 1 en busca de CVEs conocidos. Permite identificar qué repositorios y paquetes tienen más vulnerabilidades, su severidad y si tienen fix disponible.

- **Parte 2b — Análisis de código fuente (SAST)**: Analiza los resultados de CodeQL, que ejecuta queries de seguridad directamente sobre el código fuente. Permite identificar patrones problemáticos en el código como inyecciones, uso inseguro de APIs, etc.

- **Parte 3 — Análisis cuantitativo cruzado**: Combina las métricas de las tres fuentes anteriores para obtener una visión global del estado de seguridad de la organización. Incluye correlaciones entre tamaño de proyecto y vulnerabilidades, ranking de repositorios más críticos y estadísticas descriptivas.

El análisis está diseñado para ser completamente **reproducible**: cualquier persona puede clonar el repositorio, abrir el devcontainer y ejecutar todas las celdas sin configuración adicional.

## Estructura de Directorios

```
sbom-vuln-analysis/
├── data/
│   ├── repos/          # Repositorios clonados como git submodules
│   ├── repos.json      # Lista de repositorios a analizar
│   └── results/        # Resultados generados por los scripts
│       ├── *-sbom.json         # SBOMs (Syft)
│       ├── *-grype.json        # Vulnerabilidades normalizadas (Grype)
│       ├── *-grype-raw.json    # Salida original de Grype
│       └── *-codeql.json       # Issues de código (CodeQL)
├── nbs/
│   └── analysis.ipynb  # Este notebook
└── scripts/
    ├── add_submodules.py     # Clona los repos del repos.json
    ├── generate_sboms.py     # Genera SBOMs con Syft
    ├── generate_grype.py     # Escanea vulnerabilidades con Grype
    └── generate_codeql.py    # Analiza código con CodeQL
```

## Paso 0: Generación de datos

Ejecuta los scripts que generan los archivos de resultados a partir de los repositorios clonados. 

> **Nota**: La línea `#| eval: false` evita que esta celda se ejecute al hacer *Run All*, ya que la generación de datos puede tomar varios minutos. Ejecútala manualmente solo la primera vez.

In [ ]:
#| eval: false
import os
from pathlib import Path

cwd = Path().resolve()
PROJECT_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
os.chdir(PROJECT_ROOT)

!/usr/local/bin/python scripts/generate_sboms.py
!/usr/local/bin/python scripts/generate_grype.py
!/usr/local/bin/python scripts/generate_codeql.py

## Paso 1: Configuración del entorno

Importamos las librerías necesarias y configuramos las rutas base. El directorio `data/results` contiene todos los archivos JSON generados por los scripts.

In [ ]:
import os
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Fijar el directorio de trabajo en la raíz del proyecto,
# sin importar desde dónde se haya lanzado Jupyter.
# Se usa pyproject.toml como ancla para ubicar la raíz.
cwd = Path().resolve()
PROJECT_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
os.chdir(PROJECT_ROOT)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

RESULTS_DIR = PROJECT_ROOT / "data" / "results"

sbom_files   = sorted(RESULTS_DIR.glob("*-sbom.json"))
grype_files  = sorted(RESULTS_DIR.glob("*-grype.json"))
codeql_files = sorted(RESULTS_DIR.glob("*-codeql.json"))

print(f"Raíz del proyecto        : {PROJECT_ROOT}")
print(f"Directorio de resultados : {RESULTS_DIR}")
print(f"SBOMs encontrados        : {len(sbom_files)}")
print(f"Archivos Grype           : {len(grype_files)}")
print(f"Archivos CodeQL          : {len(codeql_files)}")

---
## Paso 2: Análisis de SBOMs — Parte 1

Cargamos los SBOMs generados por Syft. Cada archivo `*-sbom.json` contiene la lista completa de componentes (artefactos) detectados en un repositorio, incluyendo nombre, versión, tipo de ecosistema y lenguaje.

Con estos datos construimos dos DataFrames:
- `df_sbom`: una fila por repositorio con el total de componentes
- `df_packages`: una fila por componente con todos sus atributos

In [ ]:
sbom_records = []
all_packages = []

for sbom_file in sbom_files:
    repo_name = sbom_file.name.replace("-sbom.json", "")
    data = json.loads(sbom_file.read_text(encoding="utf-8"))
    artifacts = data.get("artifacts", [])
    ecosystems = Counter(a.get("type", "unknown") for a in artifacts)

    sbom_records.append({
        "repo": repo_name,
        "total_components": len(artifacts),
        "ecosystems": dict(ecosystems),
    })

    for artifact in artifacts:
        all_packages.append({
            "repo": repo_name,
            "package": artifact.get("name", "unknown"),
            "version": artifact.get("version", "unknown"),
            "type": artifact.get("type", "unknown"),
            "language": artifact.get("language", "unknown"),
        })

df_sbom     = pd.DataFrame(sbom_records)
df_packages = pd.DataFrame(all_packages)

print(f"Repositorios con SBOM           : {len(df_sbom)}")
print(f"Total de componentes detectados : {df_sbom['total_components'].sum():,}")
print(f"Promedio de componentes por repo: {df_sbom['total_components'].mean():.1f}")

df_sbom.sort_values("total_components", ascending=False)

### 2.1 Componentes por repositorio y distribución por ecosistema

El gráfico de barras horizontal muestra cuántos componentes tiene cada repositorio. El gráfico de barras de la derecha muestra los ecosistemas de paquetes más utilizados en toda la organización (por ejemplo, `python`, `npm`, `rust-crate`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Componentes por repositorio
top = df_sbom.nlargest(20, "total_components")
axes[0].barh(top["repo"], top["total_components"], color=sns.color_palette("muted")[0])
axes[0].set_xlabel("N° de componentes")
axes[0].set_title("Componentes por repositorio (top 20)")
axes[0].invert_yaxis()

# Ecosistemas más frecuentes
all_eco = Counter()
for row in sbom_records:
    all_eco.update(row["ecosystems"])

eco_df = pd.DataFrame(all_eco.most_common(10), columns=["ecosistema", "total"])
axes[1].bar(eco_df["ecosistema"], eco_df["total"], color=sns.color_palette("muted")[1])
axes[1].set_xlabel("Ecosistema")
axes[1].set_ylabel("N° de componentes")
axes[1].set_title("Top 10 ecosistemas de paquetes")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "sbom_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.2 Paquetes compartidos entre repositorios

Identificamos los paquetes que aparecen en más de un repositorio. Esto permite ver qué dependencias son transversales a toda la organización y cuáles podrían tener mayor impacto si presentan vulnerabilidades.

In [ ]:
top_packages = (
    df_packages.groupby(["package", "type"])
    .agg(repos_count=("repo", "nunique"))
    .reset_index()
    .nlargest(15, "repos_count")
)
print("Top 15 paquetes presentes en más repositorios:")
top_packages

---
## Paso 3: Análisis de Vulnerabilidades en Dependencias — Parte 2a

Grype tomó como entrada los SBOMs de la Parte 1 y los comparó contra su base de datos de CVEs para identificar vulnerabilidades conocidas en las dependencias.

Cada archivo `*-grype.json` contiene:
- El total de vulnerabilidades encontradas
- La distribución por severidad (critical / high / medium / low)
- El detalle de cada vulnerabilidad: paquete afectado, CVE, versión con fix

Con estos datos construimos dos DataFrames:
- `df_grype`: una fila por repositorio con el conteo de vulnerabilidades por severidad
- `df_vulns`: una fila por vulnerabilidad individual

In [ ]:
grype_records = []
all_vulns = []

for grype_file in grype_files:
    repo_name = grype_file.name.replace("-grype.json", "")
    data = json.loads(grype_file.read_text(encoding="utf-8"))
    by_sev = data.get("vulnerabilities_by_severity", {})

    grype_records.append({
        "repo"    : repo_name,
        "total"   : data.get("total_vulnerabilities", 0),
        "critical": by_sev.get("critical", 0),
        "high"    : by_sev.get("high", 0),
        "medium"  : by_sev.get("medium", 0),
        "low"     : by_sev.get("low", 0),
    })

    for vuln in data.get("vulnerabilities", []):
        all_vulns.append({
            "repo"       : repo_name,
            "package"    : vuln.get("package", "unknown"),
            "version"    : vuln.get("version", "unknown"),
            "vuln_id"    : vuln.get("vuln_id", "unknown"),
            "severity"   : vuln.get("severity", "Unknown"),
            "fix_version": vuln.get("fix_version", "N/A"),
        })

df_grype = pd.DataFrame(grype_records)
df_vulns = pd.DataFrame(all_vulns)

total_vulns = df_grype["total"].sum()
print(f"Repositorios escaneados          : {len(df_grype)}")
print(f"Total de vulnerabilidades        : {total_vulns:,}")
print(f"  Critical : {df_grype['critical'].sum():,}")
print(f"  High     : {df_grype['high'].sum():,}")
print(f"  Medium   : {df_grype['medium'].sum():,}")
print(f"  Low      : {df_grype['low'].sum():,}")

df_grype.sort_values("total", ascending=False).head(10)

### 3.1 Vulnerabilidades por repositorio y distribución global por severidad

El gráfico de barras apiladas muestra la composición de severidades por repositorio. El gráfico de torta muestra la proporción global de cada nivel de severidad en toda la organización.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

severities = ["critical", "high", "medium", "low"]
colors     = ["#d32f2f", "#f57c00", "#fbc02d", "#388e3c"]

top20 = df_grype.nlargest(20, "total").set_index("repo")
bottom = pd.Series([0] * len(top20), index=top20.index)
for sev, color in zip(severities, colors):
    if sev in top20.columns:
        axes[0].barh(top20.index, top20[sev], left=bottom, label=sev.capitalize(), color=color)
        bottom += top20[sev]

axes[0].set_xlabel("N° de vulnerabilidades")
axes[0].set_title("Vulnerabilidades por repositorio (top 20)")
axes[0].legend(loc="lower right")
axes[0].invert_yaxis()

sev_totals = {s.capitalize(): df_grype[s].sum() for s in severities if df_grype[s].sum() > 0}
if sev_totals:
    axes[1].pie(
        sev_totals.values(),
        labels=sev_totals.keys(),
        colors=[c for c, s in zip(colors, severities) if df_grype[s].sum() > 0],
        autopct="%1.1f%%",
        startangle=90,
    )
    axes[1].set_title("Distribución global por severidad")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "grype_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.2 CVEs más frecuentes y disponibilidad de fix

Identificamos los CVEs que aparecen en más repositorios (impacto transversal) y calculamos qué porcentaje de vulnerabilidades ya tiene una versión parcheada disponible.

In [ ]:
if not df_vulns.empty:
    top_cves = df_vulns["vuln_id"].value_counts().head(15).reset_index()
    top_cves.columns = ["vuln_id", "count"]

    plt.figure(figsize=(12, 4))
    plt.bar(top_cves["vuln_id"], top_cves["count"], color=sns.color_palette("muted")[2])
    plt.xlabel("CVE / ID de vulnerabilidad")
    plt.ylabel("N° de repos afectados")
    plt.title("Top 15 CVEs más frecuentes en la organización")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "top_cves.png", dpi=150, bbox_inches="tight")
    plt.show()

    con_fix = (df_vulns["fix_version"] != "N/A").sum()
    sin_fix = (df_vulns["fix_version"] == "N/A").sum()
    print(f"Vulnerabilidades con fix disponible : {con_fix:,} ({100*con_fix/len(df_vulns):.1f}%)")
    print(f"Vulnerabilidades sin fix            : {sin_fix:,} ({100*sin_fix/len(df_vulns):.1f}%)")

### 3.3 Paquetes con más vulnerabilidades

Agrupamos las vulnerabilidades por paquete y severidad para identificar cuáles dependencias concentran el mayor riesgo.

In [ ]:
if not df_vulns.empty:
    top_pkg = (
        df_vulns.groupby(["package", "severity"])
        .size()
        .unstack(fill_value=0)
        .assign(total=lambda d: d.sum(axis=1))
        .nlargest(10, "total")
    )
    print("Top 10 paquetes con más vulnerabilidades:")
    top_pkg

---
## Paso 4: Análisis de Código Fuente — Parte 2b

CodeQL construyó una base de datos de cada repositorio y ejecutó queries de seguridad sobre el código fuente directamente. A diferencia de Grype (que busca CVEs en dependencias), CodeQL detecta patrones problemáticos en el código propio del proyecto.

Cada archivo `*-codeql.json` contiene:
- El total de issues encontrados
- La distribución por nivel: `error` (crítico), `warning` (importante), `note` (informativo)
- El detalle de cada issue: regla activada, archivo afectado, mensaje

Con estos datos construimos dos DataFrames:
- `df_codeql`: una fila por repositorio con el conteo de issues por nivel
- `df_issues`: una fila por issue individual

In [ ]:
codeql_records = []
all_issues = []

for codeql_file in codeql_files:
    repo_name = codeql_file.name.replace("-codeql.json", "")
    data = json.loads(codeql_file.read_text(encoding="utf-8"))
    by_sev = data.get("issues_by_severity", {})

    codeql_records.append({
        "repo"   : repo_name,
        "total"  : data.get("total_issues", 0),
        "error"  : by_sev.get("error", 0),
        "warning": by_sev.get("warning", 0),
        "note"   : by_sev.get("note", 0),
    })

    for issue in data.get("issues", []):
        all_issues.append({
            "repo"   : repo_name,
            "rule_id": issue.get("rule_id", "unknown"),
            "level"  : issue.get("level", "unknown"),
            "file"   : issue.get("file", "unknown"),
            "message": issue.get("message", ""),
        })

df_codeql = pd.DataFrame(codeql_records)
df_issues = pd.DataFrame(all_issues)

total_issues = df_codeql["total"].sum()
print(f"Repositorios analizados  : {len(df_codeql)}")
print(f"Total de issues          : {total_issues:,}")
print(f"  Error   : {df_codeql['error'].sum():,}")
print(f"  Warning : {df_codeql['warning'].sum():,}")
print(f"  Note    : {df_codeql['note'].sum():,}")

df_codeql.sort_values("total", ascending=False).head(10)

### 4.1 Issues por repositorio y reglas más frecuentes

El gráfico de barras apiladas muestra la distribución de issues por nivel en cada repositorio. El gráfico de la derecha muestra las reglas de CodeQL que se activaron con mayor frecuencia, lo que indica los patrones de código problemático más comunes en la organización.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

levels       = ["error", "warning", "note"]
level_colors = ["#c62828", "#f9a825", "#1565c0"]

top20 = df_codeql.nlargest(20, "total").set_index("repo")
bottom = pd.Series([0] * len(top20), index=top20.index)
for level, color in zip(levels, level_colors):
    if level in top20.columns:
        axes[0].barh(top20.index, top20[level], left=bottom, label=level.capitalize(), color=color)
        bottom += top20[level]

axes[0].set_xlabel("N° de issues")
axes[0].set_title("Issues de código por repositorio (top 20)")
axes[0].legend(loc="lower right")
axes[0].invert_yaxis()

if not df_issues.empty:
    top_rules = df_issues["rule_id"].value_counts().head(10)
    axes[1].barh(top_rules.index, top_rules.values, color=sns.color_palette("muted")[3])
    axes[1].set_xlabel("N° de ocurrencias")
    axes[1].set_title("Top 10 reglas CodeQL más activadas")
    axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "codeql_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Paso 5: Análisis Cuantitativo Cruzado — Parte 3

Combinamos las métricas de las tres fuentes (SBOM, Grype, CodeQL) en un único DataFrame por repositorio. Esto nos permite correlacionar el tamaño del proyecto (componentes) con su nivel de riesgo (vulnerabilidades e issues) e identificar los repositorios que requieren mayor atención.

In [ ]:
df_combined = (
    df_sbom
    .merge(df_grype[["repo", "total"]].rename(columns={"total": "vulns"}), on="repo", how="left")
    .merge(df_codeql[["repo", "total"]].rename(columns={"total": "issues"}), on="repo", how="left")
    .fillna(0)
)

print("=" * 50)
print("RESUMEN FINAL")
print("=" * 50)
print(f"Organización             : pallets")
print(f"Repositorios analizados  : {len(df_combined)}")
print(f"Componentes totales      : {df_combined['total_components'].sum():,}")
print(f"Vulnerabilidades (Grype) : {int(df_combined['vulns'].sum()):,}")
print(f"Issues de código (CodeQL): {int(df_combined['issues'].sum()):,}")
print("=" * 50)

df_combined.sort_values("vulns", ascending=False)

### 5.1 Correlación entre componentes y vulnerabilidades

Exploramos si existe una relación entre el número de dependencias de un proyecto y la cantidad de vulnerabilidades encontradas. Un repositorio con muchas dependencias no necesariamente tiene más vulnerabilidades si sus dependencias están actualizadas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(
    df_combined["total_components"],
    df_combined["vulns"],
    alpha=0.7,
    color=sns.color_palette("muted")[0],
    s=80,
)
for _, row in df_combined.iterrows():
    if row["vulns"] > 0:
        axes[0].annotate(row["repo"], (row["total_components"], row["vulns"]),
                         fontsize=7, alpha=0.7, xytext=(4, 4), textcoords="offset points")
axes[0].set_xlabel("N° de componentes (SBOM)")
axes[0].set_ylabel("N° de vulnerabilidades (Grype)")
axes[0].set_title("Correlación: tamaño vs. vulnerabilidades")

# Repositorios más críticos
df_combined["score"] = df_combined["vulns"] + df_combined["issues"]
top_crit = df_combined.nlargest(15, "score")
x = range(len(top_crit))
axes[1].bar([i - 0.2 for i in x], top_crit["vulns"],  width=0.4, label="Vulns (Grype)",  color="#f57c00")
axes[1].bar([i + 0.2 for i in x], top_crit["issues"], width=0.4, label="Issues (CodeQL)", color="#1565c0")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(top_crit["repo"], rotation=45, ha="right")
axes[1].set_ylabel("N° de hallazgos")
axes[1].set_title("Repositorios más críticos")
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "combined_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.2 Estadísticas descriptivas

Resumen estadístico de las métricas clave por repositorio. Permite entender la distribución de componentes, vulnerabilidades e issues en toda la organización.

In [ ]:
df_combined[["total_components", "vulns", "issues"]].rename(columns={
    "total_components": "Componentes",
    "vulns"           : "Vulnerabilidades",
    "issues"          : "Issues CodeQL",
}).describe().round(2)